Assume no translation or rotation, so Extrinsic matrix is Identity

DETECTION OF CORNER POINTS IN THE CHESSBOARD

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [ ]:
def corner_pixels(filename):
  img = cv2.imread(filename)
  img_gray = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
  img_gray = np.float32(img_gray)
  corner = cv2.cornerHarris(img_gray,2,3,0.04)
  corner = cv2.dilate(corner,None)
  max = corner.max()
  count = 0
  x=0
  y=0
  points = []
  for i in range(img.shape[0]):
    for j in range(img.shape[1]):
      if(corner[i,j]>0.95*max):
        img[i,j] = [255,0,0]
        if(i-x>4 or j-y>4):
          count += 1
          points.append([i+2,j+2])
          x = i
          y = j
  points = np.array(points)
  diff = []
  for i in range(8):
    diff.append(points[i+1,1] - points[i,1])

  diff = np.array(diff)
  avg = int(np.round(np.mean(diff)))
  return img,points,avg

In [ ]:
def pixel_to_coordinate(pix,len):
  num = pix.shape[0]
  x_0 = pix[0,0] - len
  y_0 = pix[0,1] - len
  coord = np.ones((num,4))
  for i in range(num):
    coord[i,0] = pix[i,0]
    coord[i,1] = pix[i,1]
    coord[i,2] = np.round((pix[i,0] - x_0)/len)*25
    coord[i,3] = np.round((pix[i,1] - y_0)/len)*25
  print(x_0,y_0)
  return coord

In [ ]:
def ls_matrix(coord,Z):

  A = []
  for i in range(54):
    x = coord[i,0]
    y = coord[i,1]
    X = coord[i,2]
    Y = coord[i,3]

    A.append([X,Y,Z,1,0,0,0,0,-(x*X),-(x*Y),-(x*Z),-x])
    A.append([0,0,0,0,X,Y,Z,1,-(y*X),-(y*Y),-(y*Z),-y])

  A = np.array(A)
  return(A)

In [ ]:
def min_eigen_val_vec(matrix):
  e_val,e_vec=np.linalg.eig(matrix)
  min_e=min(e_val)
  min_index=np.argmin(e_val)
  min_e_vec=e_vec[:,min_index]
  return min_e,min_e_vec

In [ ]:
image1,p1,a1 = corner_pixels('c1.png')
image2,p2,a2 = corner_pixels('c2.png')
image3,p3,a3 = corner_pixels('c3.png')
image4,p4,a4 = corner_pixels('c4.png')
image5,p5,a5 = corner_pixels('c5.png')
image6,p6,a6 = corner_pixels('c6.png')
image7,p7,a7 = corner_pixels('c7.png')
image8,p8,a8 = corner_pixels('c8.png')
image9,p9,a9 = corner_pixels('c9.png')

In [ ]:
i1_c = pixel_to_coordinate(p1,a1)
i2_c = pixel_to_coordinate(p2,a2)
i3_c = pixel_to_coordinate(p3,a3)
i4_c = pixel_to_coordinate(p4,a4)
i5_c = pixel_to_coordinate(p5,a5)
i6_c = pixel_to_coordinate(p6,a6)
i7_c = pixel_to_coordinate(p7,a7)
i8_c = pixel_to_coordinate(p8,a8)
i9_c = pixel_to_coordinate(p9,a9)

201 59
263 149
305 210
334 253
357 286
373 310
387 330
399 347
408 361


In [ ]:
Z1 = 400
Z2 = 500
Z3 = 600
Z4 = 700
Z5 = 800
Z6 = 900
Z7 = 1000
Z8 = 1100
Z9 = 1200

m1 = ls_matrix(i1_c,Z1)
m2 = ls_matrix(i2_c,Z2)
m3 = ls_matrix(i3_c,Z3)
m4 = ls_matrix(i4_c,Z4)
m5 = ls_matrix(i5_c,Z5)
m6 = ls_matrix(i6_c,Z6)
m7 = ls_matrix(i7_c,Z7)
m8 = ls_matrix(i8_c,Z8)
m9 = ls_matrix(i9_c,Z9)

m = np.concatenate((m1,m2,m3,m4,m5,m6,m7,m8,m9))
A = np.transpose(m)@m
print(A)

[[ 4.60687500e+06  5.31562500e+06  3.40200000e+07  4.25250000e+04
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  -2.66974312e+09 -2.94375938e+09 -1.86777000e+10 -2.35500750e+07]
 [ 5.31562500e+06  9.61875000e+06  4.86000000e+07  6.07500000e+04
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  -2.94375938e+09 -4.92480000e+09 -2.48832000e+10 -3.11040000e+07]
 [ 3.40200000e+07  4.86000000e+07  3.43440000e+08  3.88800000e+05
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  -1.86777000e+10 -2.48832000e+10 -1.75841280e+11 -1.99065600e+08]
 [ 4.25250000e+04  6.07500000e+04  3.88800000e+05  4.86000000e+02
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  -2.35500750e+07 -3.11040000e+07 -1.99065600e+08 -2.48832000e+05]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   4.60687500e+06  5.31562500e+06  3.40200000e+07  4.25250000e+04
  -2.33574250e+09 -3.05065688e+09 -1.72680375e+10 -2.15607000e+07]
 [ 0.

In [ ]:
p = min_eigen_val_vec(A)[1]
p = p.reshape(3,4)
print(p)

[[ 6.46181783e-03  6.33939714e-07  2.32707745e-03 -5.65756670e-01]
 [ 1.58651794e-06  6.46403235e-03  2.32714141e-03 -8.24515022e-01]
 [ 2.94524368e-09  1.15983680e-09  4.54507065e-06 -9.43268468e-07]]


In [ ]:
# plt.figure(1)
# plt.imshow(image1)
# plt.figure(2)
# plt.imshow(image2)
# plt.figure(3)
# plt.imshow(image3)
# plt.figure(4)
# plt.imshow(image4)
# plt.show()

In [ ]:
X1 = np.array([[0],[0],[400],[1]])
X2 = np.array([[0],[0],[500],[1]])
X3 = np.array([[0],[0],[600],[1]])
X4 = np.array([[0],[0],[700],[1]])
X5 = np.array([[0],[0],[800],[1]])
X6 = np.array([[0],[0],[900],[1]])
X7 = np.array([[0],[0],[1000],[1]])
X8 = np.array([[0],[0],[1100],[1]])
X9 = np.array([[0],[0],[1200],[1]])


p = p.reshape(3,4)
x1 = p@X1
x2 = p@X2
x3 = p@X3
x4 = p@X4
x5 = p@X5
x6 = p@X6
x7 = p@X7
x8 = p@X8
x9 = p@X9

print(x1[0]/x1[2],x1[1]/x1[2])
print(x2[0]/x2[2],x2[1]/x2[2])
print(x3[0]/x3[2],x3[1]/x3[2])
print(x4[0]/x4[2],x4[1]/x4[2])
print(x5[0]/x5[2],x5[1]/x5[2])
print(x6[0]/x6[2],x6[1]/x6[2])
print(x7[0]/x7[2],x7[1]/x7[2])
print(x8[0]/x8[2],x8[1]/x8[2])
print(x9[0]/x9[2],x9[1]/x9[2])

[200.91207247] [58.52315354]
[263.15554992] [149.25905578]
[304.64402251] [209.73919197]
[334.27513031] [252.93416454]
[356.49653893] [285.32759181]
[373.77871778] [310.52081935]
[387.60374337] [330.67435547]
[398.91465361] [347.16292082]
[408.34008605] [360.90291659]


In [ ]:
E = np.array([[1,0,0,0],[0,1,0,0],[0,0,1,0]])
E_pseudo = (np.transpose(E))@(np.linalg.inv(E@np.transpose(E)))
k = p@E_pseudo
print(k)

[[6.46181783e-03 6.33939714e-07 2.32707745e-03]
 [1.58651794e-06 6.46403235e-03 2.32714141e-03]
 [2.94524368e-09 1.15983680e-09 4.54507065e-06]]
